In [1]:
import pandas as pd
import plotnine as pn
import numpy as np
from scipy.stats import beta
from itables import init_notebook_mode
import polars as pl
import os
import glob
from mizani.formatters import custom_format

init_notebook_mode(all_interactive=True)
from scipy.stats import mannwhitneyu


In [2]:
import sys
sys.path.append("../")   # path to folder containing the python file

from utils.load_gtf_cgc_dresden import *
from ProteinExpression.load_pr_data import *
from utils.util_functions import *

# VEP

In [43]:
somatic_snvs = pl.scan_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/snv.tsv.gz", separator="\t", infer_schema_length=1000000).select(["seqnames", "start", "end", "Ref", "Alt"]).collect().to_pandas()

somatic_snvs = somatic_snvs.drop_duplicates(subset=["seqnames", "start", "end", "Ref", "Alt"])
somatic_snvs = somatic_snvs.rename(columns={"seqnames": "#CHROM", "start": "POS", "Ref": "REF", "Alt": "ALT"})

somatic_snvs = somatic_snvs.sort_values(by=["#CHROM", "POS"])

In [4]:
def format_for_vep(row):
    chrom = str(row['#CHROM']).replace('chr', '')
    pos = int(row['POS'])
    ref = str(row['REF'])
    alt = str(row['ALT'])
    
    # 1. Handle SNVs
    if len(ref) == len(alt) == 1:
        return pd.Series([chrom, pos, pos, f"{ref}/{alt}", "+"])
    
    # 2. Handle Insertions (VCF style: ref=T, alt=TTAAG)
    elif len(alt) > len(ref):
        # Remove the padding base
        inserted_sequence = alt[len(ref):]
        # Start = Pos + 1, End = Pos
        return pd.Series([chrom, pos + 1, pos, f"-/{inserted_sequence}", "+"])
    
    # 3. Handle Deletions (VCF style: ref=TAG, alt=T)
    elif len(ref) > len(alt):
        # The deleted sequence starts after the padding base
        deleted_sequence = ref[len(alt):]
        start = pos + 1
        end = pos + len(deleted_sequence)
        return pd.Series([chrom, start, end, f"{deleted_sequence}/-", "+"])

In [42]:
indices = np.linspace(0, len(somatic_snvs), 6, dtype=int)

for k in range(len(indices) - 1):
    start_idx = indices[k]
    end_idx = indices[k+1]
    
    vep_df = somatic_snvs[start_idx: end_idx].apply(format_for_vep, axis=1)
    vep_df.columns = ['chrom', 'start', 'end', 'allele', 'strand']
    vep_df["start"] = vep_df["start"] - 1
    vep_df["end"] = vep_df["end"] - 1
    vep_df["identifier"] = vep_df["chrom"] + ":" +  vep_df["start"].astype(str) + ":" + vep_df["allele"]
    vep_df.to_csv(f"/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/vep/somatic/somatic_master_object_snvs_chunk{k}.txt", sep="\t", index=None, header=None)


In [5]:
input_df = pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_predisp_cgc_all_TINDA.parquet").select(["#CHROM", "POS", "REF", "ALT"]).collect().to_pandas()

input_df = input_df.drop_duplicates(subset=["#CHROM", "POS", "REF", "ALT"])
input_df.shape

(2011140, 4)

In [6]:

indices = np.linspace(0, len(input_df), 15, dtype=int)

for k in range(len(indices) - 1):
    start_idx = indices[k]
    end_idx = indices[k+1]
    
    vep_df = input_df[start_idx: end_idx].apply(format_for_vep, axis=1)
    vep_df.columns = ['chrom', 'start', 'end', 'allele', 'strand']
    vep_df["start"] = vep_df["start"]
    vep_df["end"] = vep_df["end"]
    vep_df["identifier"] = vep_df["chrom"] + ":" +  vep_df["start"].astype(str) + ":" + vep_df["allele"]
    vep_df.to_csv(f"/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/vep/germline_predisp_cgc_all_TINDA_chunk{k}.txt", sep="\t", index=None, header=None)

    

In [4]:
vep_somatic_res_dir = "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/vep/somatic/output"
all_res_somatic = []
for f in os.listdir(vep_somatic_res_dir):
    current_df = pl.read_csv(f"{vep_somatic_res_dir}/{f}", separator="\t",  comment_prefix="##", infer_schema=False)
    current_df = create_max_spliceai_vep_ploras(current_df).to_pandas()
    current_df = unique_vep_res_per_gene(current_df)
    all_res_somatic.append(current_df)
    
all_res_somatic = pd.concat(all_res_somatic)

all_res_somatic

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [5]:
all_res_somatic.to_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/snv_somatic_predips_cgc_vep.parquet", index=None)

In [84]:
somatic_snvs  = pl.scan_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/snv.tsv.gz", separator="\t", infer_schema=False).collect(engine="streaming").to_pandas()
somatic_snvs["start"] = somatic_snvs["start"].astype(int)
somatic_vep_res = pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/snv_somatic_predips_cgc_vep.parquet",).collect(engine="streaming").to_pandas()


In [158]:
somatic_vep_res["chrom"] = somatic_vep_res["Location"].str.split(":").str[0]
somatic_vep_res["POS"] = 0
somatic_vep_res.loc[somatic_vep_res["VARIANT_CLASS"] == "SNV", "POS"] = somatic_vep_res.loc[somatic_vep_res["VARIANT_CLASS"] == "SNV", "Location"].str.split(":").str[1].astype(int)
somatic_vep_res["POS"] = somatic_vep_res["POS"] + 1

somatic_vep_res.loc[somatic_vep_res["VARIANT_CLASS"] != "SNV", "POS"] = somatic_vep_res.loc[somatic_vep_res["VARIANT_CLASS"] != "SNV", "#Uploaded_variation"].str.split(":").str[1].astype(int)






In [ ]:
somatic_snvs = somatic_snvs.merge(somatic_vep_res, left_on=["seqnames",  "start", "Ref", "Alt", "Gene"], right_on=["chrom", "POS", "REF_ALLELE", "Allele", "SYMBOL"], how="left") 
somatic_snvs = somatic_snvs.drop(columns=["__index_level_0__"])
somatic_snvs["sampleID"] = somatic_snvs["group_name"].str.split(".").str[1]

somatic_snvs.to_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/somatic_snvs_vep.parquet", index=None)

In [177]:
somatic_snvs = pd.read_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/somatic_snvs_vep.parquet")
somatic_snvs["sampleID"] = somatic_snvs["group_name"].str.split(".").str[1]

valid_positions = (
    somatic_snvs["POS"]
    .dropna()
    .unique()
    .astype(int)
    .tolist()
)
somatic_snvs = somatic_snvs.rename(columns={"Location_x": "functional_location", "Gene_x": "MASTER_annotated_gene", "Location_y": "vep_location", "Gene_y": "vep_Gene"})
somatic_snvs.shape


(704698, 105)

In [178]:
absplice_predisp = (pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/resources/predisp_cgc_max_abSplice_snvs_hg19.parquet")
        .filter((pl.col("hg19_end").is_in(valid_positions)))
        .collect(engine="streaming")
      ).to_pandas()
absplice_predisp["hg19_chrom"] = absplice_predisp["hg19_chrom"].str[3:]
absplice_predisp.shape



(13191, 17)

In [179]:
somatic_snvs = somatic_snvs.merge(absplice_predisp, left_on=["seqnames", "POS", "Ref", "Alt", "vep_Gene"], right_on=["hg19_chrom", "hg19_end", "ref", "alt", "gene_id"], how="left")
print(somatic_snvs.shape)
somatic_snvs = somatic_snvs.drop(columns=["chrom_x", "start_y", "end_y"]).rename(columns={"start_x": "start", "end_x": "end"})

(704698, 122)


In [180]:
abexp = pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/resources/predisp_cgc_max_abexp.parquet").filter((pl.col("start").is_in(valid_positions)) | (pl.col("end").is_in(valid_positions))).collect(engine="streaming").to_pandas()

abexp["seqnames"] = abexp["chrom"].str.split("chr").str[1]

promoter_ai = (pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/resources/promoterAI_tss500_hg19.parquet")
                .filter(pl.col("gene").is_in(somatic_snvs["MASTER_annotated_gene"]))
                .filter(pl.col("hg19_start").is_in(valid_positions))
                .collect(engine="streaming").to_pandas()
              )
df_sorted = promoter_ai.sort_values(
    by="promoterAI",
    key=lambda x: x.abs(),
    ascending=False
)
promoter_ai = df_sorted.drop_duplicates(
    subset=["chrom", "pos", "ref", "alt", "gene_id", "strand"],
    keep="first"
)
promoter_ai["seqnames"] = promoter_ai["chrom"].str[3:]
promoter_ai = promoter_ai.drop(columns="chrom")



In [181]:
merged_vars = somatic_snvs.merge(promoter_ai, left_on=["seqnames", "POS", "Ref", "Alt", "vep_Gene"], right_on=["seqnames", "hg19_start", "ref", "alt", "gene_id"], how="left")      

merged_vars = merged_vars.merge(abexp, left_on=["POS", "seqnames", "Ref", "Alt", "vep_Gene"], right_on=["end", "seqnames", "ref", "alt", "gene"], how="left")


In [187]:
merged_vars = merged_vars.rename(columns={"#Uploaded_variation": "somatic_snv_#Uploaded_variation", "Consequence": "somatic_snv_Consequence", "IMPACT": "somatic_snv_IMPACT", "am_pathogenicity": "somatic_snv_am_pathogenicity","LoF": "somatic_snv_am_LoF", "max_spliceai_score": "somatic_snv_max_spliceai_score", "pangolin_score": "somatic_snv_pangolin_score", "AbSplice2_max": "somatic_snv_AbSplice2_max", "promoterAI": "somatic_snv_promoterAI", "abexp_v1.1": "somatic_snv_abexp_v1.1" })
merged_vars.to_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/somatic_snvs_vep_annotated.parquet", index=None)
                                          

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [4]:
vep_res_dir = "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/vep/outputs"
all_res = []
for f in os.listdir(vep_res_dir):
    if not "warning" in f:
        current_df = pl.read_csv(f"{vep_res_dir}/{f}", separator="\t",  comment_prefix="##", infer_schema=False)
        current_df = create_max_spliceai_vep_ploras(current_df).to_pandas()
        current_df = unique_vep_res_per_gene(current_df)
        all_res.append(current_df)
                       
all_res = pd.concat(all_res)

In [5]:
all_res.to_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_predisp_cgc_all_TINDA_vep.parquet", index=None)

In [38]:
germline_snvs  = pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_predisp_cgc_all_TINDA.parquet").head(100).collect(engine="streaming").to_pandas()
germline_vep_res = pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_predisp_cgc_all_TINDA_vep.parquet",).collect(engine="streaming").to_pandas()

germline_snvs[["REF_trimmed", "ALT_trimmed"]] = germline_snvs.apply(left_align_trim_vcf, axis=1)


In [39]:
germline_snvs.loc[(germline_snvs["REF"].str.len() > 1) | (germline_snvs["ALT"].str.len() > 1), "ALT"] = germline_snvs.loc[(germline_snvs["REF"].str.len() > 1) | (germline_snvs["ALT"].str.len() > 1), "ALT_trimmed"]
germline_snvs.loc[(germline_snvs["REF"].str.len() > 1) | (germline_snvs["ALT"].str.len() > 1), "REF"] = germline_snvs.loc[(germline_snvs["REF"].str.len() > 1) | (germline_snvs["ALT"].str.len() > 1), "REF_trimmed"]

In [40]:
germline_vep_res["chrom"] = germline_vep_res["Location"].str.split(":").str[0]
germline_vep_res["POS"] = 0
germline_vep_res.loc[germline_vep_res["VARIANT_CLASS"] == "SNV", "POS"] = germline_vep_res.loc[germline_vep_res["VARIANT_CLASS"] == "SNV", "Location"].str.split(":").str[1].astype(int)
germline_vep_res["POS"] = germline_vep_res["POS"] + 1

germline_vep_res.loc[germline_vep_res["VARIANT_CLASS"] != "SNV", "POS"] = germline_vep_res.loc[germline_vep_res["VARIANT_CLASS"] != "SNV", "#Uploaded_variation"].str.split(":").str[1].astype(int)





In [45]:
germline_snvs = germline_snvs.merge(germline_vep_res, left_on=["#CHROM",  "POS", "REF", "ALT", "HUGO_Symbol"], right_on=["chrom", "POS", "REF_ALLELE", "Allele", "SYMBOL"], how="left") 


Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


# Create Tinda vcfs

In [ ]:
import sys
sys.path.append("/home/a379i/Scripts")   # path to folder containing the python file

from utils.load_gtf_cgc_dresden import *
from ProteinExpression.load_pr_data import *

In [4]:
sa = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/sample_data/master_drop_sample_annotation_sizeFactorFiltered_0.1.tsv", sep="\t")

sample_ids = sa[sa["Diag"] != "Unstranded_data"]["pid"]


In [6]:
all_vars  = pd.read_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_predisp_all_TINDA.parquet")




In [7]:
# all_dfs = pd.concat(all_dfs)
columns = ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'FORMAT',
       'control_raw', 'tumor_raw', 'control_dp', 'tumor_dp', 'control_ad',
       'tumor_ad', 'control_vaf', 'tumor_vaf', 'gt_class', 'control_gt',
       'tumor_gt', 'control_zygosity', 'tumor_zygosity', 'vep_gene_name', 'vep_consequence', 'vep_impact', "sampleID"]
def process_vcf(file_path, sample, control_type, tumor_type, vcf_type="tinda"):
    try:
        df = pl.read_csv(file_path, separator="\t", comment_prefix="##",  schema_overrides={
                                    "QUAL": pl.String,
                                    "#CHROM": pl.String,
                                    "RNA_VARIANT_AF": pl.Float64,
                                }, 
                    null_values=["NA", "."],
                )
    except Exception(e):
        print(e)
        print('no germline vcf for sample ' , file_path)
        print(sample, control_type, tumor_type)
        return pd.DataFrame()
    # print(df.shape)
    samples = [f"sample_{control_type}_{sample}", f"sample_{tumor_type}_{sample}"]

    df = df.rename({
            df.columns[9]: "control_raw",
            df.columns[10]: "tumor_raw"
        })
    
    # for i, s in enumerate(samples):
        
    
    #     col_name = "control" if i == 0 else "tumor"
    #     df = df.with_columns([
    #         pl.col(s).str.split(":").list.get(0).alias(f"{col_name}_gt") 
    #     ])

    # # Loop 2: Calculate Zygosity
    # for i, s in enumerate(samples):
    #     col_name = "control" if i == 0 else "tumor"
    #     df = df.with_columns([
    #         pl.when(pl.col(f"{col_name}_gt").str.replace(r"\|", "/").is_in(["0/1", "1/0"]))
    #           .then(pl.lit("Het"))
    #           .when(pl.col(f"{col_name}_gt").str.replace(r"\|", "/").is_in(["1/1"]))
    #           .then(pl.lit("Hom Alt"))
    #           .when(pl.col(f"{col_name}_gt").str.replace(r"\|", "/").is_in(["0/0"]))
    #           .then(pl.lit("Hom Ref"))
    #           .otherwise(pl.lit("Unknown/Other"))
    #           .alias(f"{col_name}_zygosity")
    #     ])

    
    #     df = df.rename({
    #         samples[0]: "control_raw",
    #         samples[1]: "tumor_raw"
    #     })

    if vcf_type == "tinda":
        df = df.filter(pl.col("Control_VAF") >= 0.1)
        df = df.filter((pl.col("HUGO_Symbol").is_in(extended_dresden_list)) | (pl.col("HUGO_Symbol").is_in(cgc["GENE_SYMBOL"])))
        # print(df.shape)

    else:
        df = df.with_columns([
        # Extract Depths (DP)
        pl.col("INFO").str.extract(r"Control_dp=(\d+)", 1).cast(pl.Int64).alias("control_dp"),
        pl.col("INFO").str.extract(r"Tumor_dp=(\d+)", 1).cast(pl.Int64).alias("tumor_dp"),
        
        # Extract Alt Depths (AD)
        pl.col("INFO").str.extract(r"Control_dpALT=(\d+)", 1).cast(pl.Int64).alias("control_ad"),
        pl.col("INFO").str.extract(r"Tumor_dpALT=(\d+)", 1).cast(pl.Int64).alias("tumor_ad"),
        
        # Extract VAFs
        pl.col("INFO").str.extract(r"Control_VAF=([0-9.]+)", 1).cast(pl.Float64).alias("control_vaf"),
        pl.col("INFO").str.extract(r"Tumor_VAF=([0-9.]+)", 1).cast(pl.Float64).alias("tumor_vaf"),
        
        # Extract Classification for context
        pl.col("INFO").str.extract(r"GT_Classification=([^;]+)", 1).alias("gt_class")
        ])
            
 
    
    
        # Assuming your column is named "info"
        df = df.with_columns(
            vep_gene_name = pl.col("INFO").str.extract(r"CSQ=[^|]*\|[^|]*\|[^|]*\|([^|]+)")
        )
    
        df = df.with_columns(
            # 1. Pull the raw CSQ string out of the INFO column
            csq_raw = pl.col("INFO").str.extract(r"CSQ=([^;]+)", 1)
            ).with_columns(
                # 2. Split by comma to get the first transcript entry, then split by pipe
                csq_parts = pl.col("csq_raw").str.split(",").list.get(0).str.split("|")
            ).with_columns(
                # 3. Assign the specific indices to new columns
                vep_consequence = pl.col("csq_parts").list.get(1),
                vep_impact      = pl.col("csq_parts").list.get(2),
            )
    
        df = df.filter(pl.col("vep_gene_name").is_in(extended_dresden_list))
    # print(df.shape)
    df = df.to_pandas()
    df["sampleID"] = sample.split("-")[1]
    
    # ONLY convert to pandas at the very end of the function
    #return df[columns]
    return df

In [8]:
## sample: 1688 problematic
## sample: 3501 problematic sample_buffy-coat2_K26K-V3TLD6
## sample 3980 sample_blood04_H021-SD8H8Q
# 3587: sample_buffy-coat1-01_K26K-WFT5H9 probaby there are both samples
sa = sa[(~sa["pid"].isin(all_vars["sampleID"].unique()))]
sa

NameError: name 'all_vars' is not defined

In [9]:
all_dfs = []
full_paths = []

for i, row in sa.iterrows():
    if i % 500 == 0:
        print(i)
    # if i == 1688 or i == 3501 or i == 3980 or i == 3587:
    #     continue
    sample = row["RNA_BAM_FILE"].split("/")[-1].split("_")[1]
    sample_type = row["sample_type"]



    # Define the potential base paths (WGS vs Exon)
    base_dirs = [
        f"/omics/odcf/analysis/hipo/hipo_021/GermlineAnalysis/data_object_master_germline/sequencing/whole_genome_sequencing/results_per_pid/{sample}/",
        f"/omics/odcf/analysis/hipo/hipo_021/GermlineAnalysis/data_object_master_germline/sequencing/exon_sequencing/results_per_pid/{sample}/"
    ]
    
    file_path = None
    control_type = None
    extracted_sample_type = None

    # Search strategy: Try every base dir, then every version, then every tissue suffix
    ## order: whole_genome v5, whole_genome_v3, wes_v5, wes_v3
    for b_path in base_dirs:
        if not os.path.exists(b_path):
            continue
        # Use a wildcard '*' to catch .blood, .blood02, .buffy_coat, etc.
        # We also check both v3.0.0 and v5 in one go
        # pattern = os.path.join(b_path, "germline_smallVariants.*", "v*", f"smallVariants_{sample}.clean_annotated.rare.VEP.vcf.gz")
        pattern = os.path.join(b_path, "germline_smallVariants.*", "v*", "clinvar*", f"smallVariants_{sample}.clean_annotated.rare.VEP.CharGer.TiNDA.rare_germline.vcf")

        # print(pattern)

        matches = glob.glob(pattern)
        
        
        if matches:
            # Take the latest version/match
            file_path = sorted(matches)[-1] 

            # Path structure: .../germline_smallVariants.TISSUE.CONTROL/v3.0.0/...
            path_parts = file_path.split('/')
            variant_folder = path_parts[-3] # e.g., "germline_smallVariants.blood.blood"
            
            # Split by '.' 
            # Parts will be: ["germline_smallVariants", "sample_type", "control_type"]
            folder_bits = variant_folder.split('.')
            
            if len(folder_bits) < 3:
                extracted_sample_type = row["sample_type"]
                control_type = folder_bits[-1]

            else:                       
                extracted_sample_type = folder_bits[1]
                control_type = folder_bits[2]
            break
        else:
            pattern = os.path.join(b_path, "germline_smallVariants.*", "v*", f"smallVariants_{sample}.clean_annotated.rare.VEP.CharGer.TiNDA.rare_germline.vcf")
    
            matches = glob.glob(pattern)
            
            if matches:
                # Take the latest version/match
                file_path = sorted(matches)[-1] 
    
                # Path structure: .../germline_smallVariants.TISSUE.CONTROL/v3.0.0/...
                path_parts = file_path.split('/')
                variant_folder = path_parts[-3] # e.g., "germline_smallVariants.blood.blood"
                
                # Split by '.' 
                # Parts will be: ["germline_smallVariants", "sample_type", "control_type"]
                folder_bits = variant_folder.split('.')
                
                if len(folder_bits) < 3:
                    extracted_sample_type = row["sample_type"]
                    control_type = folder_bits[-1]
    
                else:                       
                    extracted_sample_type = folder_bits[1]
                    control_type = folder_bits[2]
                break
            
                

    if file_path:
        # print(f"Processing: {file_path}")
        df = process_vcf(file_path, sample, control_type, extracted_sample_type)
        # print(file_path)
        full_paths.append(file_path)
        all_dfs.append(df)
    else:
        # print(pattern, "here")
        print(f"Warning: No file found for sample {sample}")
        continue
all_dfs = pd.concat(all_dfs)

all_dfs['max_gnomAD_AF'] = pd.to_numeric(all_dfs['max_gnomAD_AF'], errors='coerce')
all_dfs['max_gnomAD_AC'] = pd.to_numeric(all_dfs['max_gnomAD_AC'], errors='coerce')
all_dfs['max_gnomAD_HOMO'] = pd.to_numeric(all_dfs['max_gnomAD_HOMO'], errors='coerce')
all_dfs['max_LC_VF'] = pd.to_numeric(all_dfs['max_LC_VF'], errors='coerce')
all_dfs['RNA_VARIANT_AF'] = pd.to_numeric(all_dfs['RNA_VARIANT_AF'], errors='coerce')
all_dfs['CLNILinMASTER'] = pd.to_numeric(all_dfs['CLNILinMASTER'], errors='coerce')


all_dfs.to_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_predisp_cgc_all_TINDA.parquet", index=None)

0
500
1000
1500
2000
2500
3000
3500
4000


In [11]:
all_dfs["HUGO_Symbol"].unique()

<ArrowStringArray>
[   'RPL22',     'PAX7',    'PTCH2',    'EPAS1',     'MSH6',    'ERBB4',
   'FANCD2',    'SETD2',   'COL7A1',     'POLQ',
 ...
    'NOP10',    'TREX1',     'MYCN',    'SHTN1',   'NUTM2B', 'C15orf65',
    'HOXA9',    'PCBP1',     'SSX4',     'NSD2']
Length: 855, dtype: str

In [14]:




# loaded_res = pd.read_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_predisp_all_TINDA.parquet")

# loaded_res = pd.concat([loaded_res, all_dfs])

all_dfs.to_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_predisp_all_TINDA.parquet", index=None)

In [66]:
base_path = "/omics/odcf/analysis/hipo/hipo_021/GermlineAnalysis/data_object_master_germline/sequencing/whole_genome_sequencing/results_per_pid/H021-1174YV/germline_smallVariants.metastasis02.blood/v3.0.0"
tinda_res = pl.read_csv(f"{base_path}/smallVariants_H021-1174YV.clean_annotated.rare.VEP.CharGer.TiNDA.rare_germline.vcf", separator="\t", comment_prefix="##",  schema_overrides={
                                    "QUAL": pl.String,
                                    "#CHROM": pl.String,
                                }, 
                    null_values=["NA", "."],
                )

tinda_res = tinda_res.filter((pl.col("TiN_Class") == "Germline") & (pl.col("Control_VAF") >= 0.1))
tinda_res.filter(pl.col("HUGO_Symbol").is_in(extended_dresden_list))
tinda_res["HUGO_Symbol"].unique()

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [145]:
base_path = "/omics/odcf/analysis/hipo/hipo_021/GermlineAnalysis/data_object_master_germline/sequencing/whole_genome_sequencing/results_per_pid/H021-1174YV/germline_smallVariants.metastasis02.blood/v3.0.0"
tinda_res = pl.read_csv(f"{base_path}/smallVariants_H021-1174YV.clean_annotated.rare.VEP.CharGer.TiNDA.rare_germline.vcf", separator="\t", comment_prefix="##",  schema_overrides={
                                    "QUAL": pl.String,
                                    "#CHROM": pl.String,
                                }, 
                    null_values=["NA", "."],
                )

# tinda_res = tinda_res.filter((pl.col("TiN_Class") == "Germline") & (pl.col("Control_VAF") >= 0.1))
# tinda_res = tinda_res.filter((pl.col("TiN_Class") == "Germline") )

tinda_res = tinda_res.filter(pl.col("HUGO_Symbol").is_in(extended_dresden_list))
print(tinda_res.shape)
tinda_res["TiN_Class"].value_counts()



(269, 53)


Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [157]:
base_path = "/omics/odcf/analysis/hipo/hipo_021/GermlineAnalysis/data_object_master_germline/sequencing/whole_genome_sequencing/results_per_pid/H021-1174YV/germline_smallVariants.metastasis02.blood/v3.0.0"
file_path = f"{base_path}/smallVariants_H021-1174YV.clean_annotated.rare.VEP.vcf.gz"
vep_res = process_vcf(file_path, "H021-1174YV", "blood", "metastasis02", vcf_type="raw_vcf")
# vep_res = vep_res[(~vep_res["gt_class"].str.contains("Somatic")) & (vep_res["control_vaf"] >= 0.1) & (vep_res["control_ad"] >= 5)]
vep_res = vep_res[(~vep_res["gt_class"].str.contains("Somatic")) ]

vep_res['BRF'] = pd.to_numeric(vep_res['INFO'].str.extract(r'BRF=([\d\.]+)', expand=False))
vep_res['HP'] = pd.to_numeric(vep_res['INFO'].str.extract(r'HP=([\d\.]+)', expand=False))
vep_res['FR'] = pd.to_numeric(vep_res['INFO'].str.extract(r'FR=([\d\.]+)', expand=False))
vep_res['TC'] = pd.to_numeric(vep_res['INFO'].str.extract(r'TC=([\d\.]+)', expand=False))

vep_res =vep_res[(vep_res["BRF"] <= 0.25) & (vep_res["HP"] <= 5) & ( ((vep_res["FR"] <= 0.6) & (vep_res["FR"] >= 0.4)) | (vep_res["FR"] >= 0.9)) & (vep_res["TC"] >=30)]

vep_res[["#CHROM", "POS", "REF", "ALT", "gt_class", "vep_gene_name", "FILTER", "FR", "HP", "BRF", "TC", "tumor_vaf", "control_vaf", "INFO"]]
# vep_res["gt_class"].shape

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [153]:
vep_res.columns

Index(['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT',
       'control_raw', 'tumor_raw', 'control_dp', 'tumor_dp', 'control_ad',
       'tumor_ad', 'control_vaf', 'tumor_vaf', 'gt_class', 'vep_gene_name',
       'csq_raw', 'csq_parts', 'vep_consequence', 'vep_impact', 'sampleID',
       'BRF', 'HP', 'FR', 'TC'],
      dtype='str')

In [143]:
merged = tinda_res.to_pandas().merge(vep_res, on=["#CHROM", "POS", "REF", "ALT",], how="outer")
merged[merged["gt_class"].isna()].head()

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [144]:


merged[merged["HUGO_Symbol"].isna()][['ID_y', 'QUAL_y', "BRF", "HP", "FR", "TC",
       'FILTER_y', 'INFO_y', 'FORMAT_y', 'control_raw', 'tumor_raw',
       'control_dp', 'tumor_dp', 'control_ad', 'tumor_ad', 'control_vaf',
       'tumor_vaf', 'gt_class', 'vep_gene_name', 'csq_raw', 'csq_parts',
       'vep_consequence', 'vep_impact', 'sampleID']].head()

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [61]:
absplice_predisp = (pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/resources/predisp_max_abSplice_snvs_hg19.parquet")
        .filter(pl.col("hg19_start").is_in(valid_positions))
        .collect(engine="streaming")
      ).to_pandas()
absplice_predisp.sort_values("AbSplice2_max", Ascending=False)

np.float64(0.5147082805633545)

In [11]:
# Assuming your column is named "info"
df = df.with_columns(
    vep_gene_name = pl.col("INFO").str.extract(r"CSQ=[^|]*\|[^|]*\|[^|]*\|([^|]+)")
)

In [40]:
df = df.with_columns(
    # 1. Pull the raw CSQ string out of the INFO column
    csq_raw = pl.col("INFO").str.extract(r"CSQ=([^;]+)", 1)
).with_columns(
    # 2. Split by comma to get the first transcript entry, then split by pipe
    csq_parts = pl.col("csq_raw").str.split(",").list.get(0).str.split("|")
).with_columns(
    # 3. Assign the specific indices to new columns
    vep_consequence = pl.col("csq_parts").list.get(1),
    vep_impact      = pl.col("csq_parts").list.get(2),
    gene_symbol     = pl.col("csq_parts").list.get(3)
)

In [51]:
df.filter((pl.col("HUGO_Symbol").is_in(extended_dresden_list)) | (pl.col("vep_gene_name").is_in(extended_dresden_list)))




ColumnNotFoundError: unable to find column "HUGO_Symbol"; valid columns: ["#CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT", "sample_blood_H021-3KL9JT", "sample_tumor02_H021-3KL9JT", "control_dp", "tumor_dp", "control_ad", "tumor_ad", "control_vaf", "tumor_vaf", "gt_class", "sample_blood_H021-3KL9JT_gt", "sample_tumor02_H021-3KL9JT_gt", "sample_blood_H021-3KL9JT_zygosity", "sample_tumor02_H021-3KL9JT_zygosity", "csq_raw", "csq_first", "vep_consequence", "vep_impact", "csq_parts", "gene_symbol"]

In [38]:
germline_snvs = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_snv_hg38.tsv", sep="\t")
valid_positions = (
    germline_snvs["hg38_pos"]
    .dropna()
    .unique()
    .astype(int)
    .tolist()
)



In [71]:
germline_snvs[(germline_snvs["group_name"] == "WGS.3KL9JT.tumor02") & (germline_snvs["seqnames"] == "1")]

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [ ]:
sample_master = germline_snvs[germline_snvs["group_name"] == "WGS.3KL9JT.tumor02"]


In [ ]:
len(set(germline_snvs[germline_snvs["group_name"] == "WGS.3KL9JT.tumor02"]["start"]))

In [89]:
merged = sample_master.merge(df.to_pandas(), left_on=["start", "seqnames", "Ref", "Alt"], right_on=["POS", '#CHROM', "REF", "ALT"])

In [105]:
merged[["start", "POS", "Ref", "Alt", "Location", "vep_gene_name", "Gene", "VEP_Most_Severe_Consequence"]]

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [103]:
df.filter(pl.col("VEP_Most_Severe_Consequence") == "intron_variant")

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [ ]:
extended_dresden_list

In [5]:
import polars as pl

snvs = (
    pl.scan_csv(
        "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/vep_res_rare_snv_all_aggregated.tsv", 
        separator="\t"
    )
    # 1. Broad filter ONLY for Gene and Sample first
    .filter(
        (pl.col("sampleID") == "3KL9JT")
    )
    .filter(
        (pl.col("SYMBOL").is_in(extended_dresden_list))
    ).head(100)
    .collect(engine="streaming")
)


In [19]:
snvs = snvs.unique("#Uploaded_variation")


In [24]:
snvs["Location"]

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [14]:
df.filter(pl.col("POS") == 10300779)

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [25]:
df[["#CHROM", "POS"]]

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)
